# 环境监测与短信报警一体化系统

## 项目简介

本项目实现一个将环境监测、本地报警和短信告警整合在一起的综合原型系统。Arduino 周期采集 DHT22、MQ-2 和水位传感器数据，根据烟雾、高温和低水位等异常情况做出分级判断，并通过蜂鸣器、本地 LCD 和 SIM800L 向预设手机号码发送短信告警。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| Arduino Uno | 1 | 主控板 |
| DHT22 | 1 | 温湿度采集 |
| MQ-2 | 1 | 烟雾 / 可燃气体检测 |
| 水位传感器 | 1 | 水位高低判断 |
| SIM800L | 1 | 发送短信 |
| I2C LCD1602 | 1 | 本地数据显示 |
| 蜂鸣器 | 1 | 本地声报警 |


## 核心功能

- 周期采集温湿度、烟雾值和水位值。
- 按阈值判断是否进入正常、预警或严重告警状态。
- 本地 LCD 轮播显示环境数据和当前告警级别。
- SIM800L 在状态变化时发送短信，并支持短信查询当前状态。
- 通过冷却时间限制重复短信，避免异常持续时频繁刷屏。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| DHT22 | D4 |
| MQ-2 模拟输出 | A0 |
| 水位传感器 | A1 |
| 蜂鸣器 | D8 |
| SIM800L RX / TX | D10、D11 |
| LCD SDA / SCL | I2C 总线 |


## 接线说明

- DHT22 数据脚接 D4，MQ-2 模拟输出接 A0，水位传感器模拟输出接 A1。
- SIM800L 通过 `SoftwareSerial` 与 Arduino 通信，需注意电源电流能力和电平兼容。
- LCD1602 使用 I2C 接口接入主控，地址默认 `0x27`。
- 蜂鸣器接 D8 用于本地声报警。
- SIM800L 建议使用稳定外部电源，避免发短信瞬间电流过大导致模块重启。


## 短信协议与交互

- 上电初始化后，程序会向 SIM800L 发送 `AT`、`AT+CMGF=1` 和短信上报配置命令。
- 当状态发生变化时，程序会向预设手机号发送一条包含温湿度、烟雾值、水位值和状态的短信。
- 用户发送 `STATUS` 到模块号码时，系统会回发当前环境状态短信。
- 通过 `SMS_COOLDOWN_MS` 控制重复告警的最小发送间隔。


## 运行方式

1. 上传 `src/environmental_monitoring_and_sms_alert_integrated_system/environmental_monitoring_and_sms_alert_integrated_system.ino`。
2. 完成 DHT22、MQ-2、水位传感器、蜂鸣器、LCD 和 SIM800L 接线。
3. 修改代码中的手机号和各类阈值，使其适合实际测试环境。
4. 先用串口观察采样值，再模拟烟雾、高温或低水位异常。
5. 使用手机发送 `STATUS`，验证短信查询与回复功能。


## 控制逻辑说明

- 系统维护 `NORMAL / WARNING / CRITICAL` 三种告警级别。
- 烟雾、高温或低水位中的任意单项异常会进入预警状态。
- DHT 故障或烟雾与低水位同时异常会进入严重告警状态。
- 程序只在状态切换或超出重复发送冷却时间时发送短信，避免持续抖动时刷屏。
- 本地蜂鸣器会根据预警和严重告警使用不同节奏，便于不看屏幕时判断风险级别。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 环境采集 | LCD 能稳定显示温湿度、烟雾和水位数值 |
| 分级告警 | 单项异常与多项异常能进入不同告警级别 |
| 短信发送 | 状态变化时手机能收到告警短信 |
| 短信查询 | 发送 `STATUS` 后系统能回复当前状态 |


## 可扩展方向

- 增加 GPS 定位，将告警位置一并发送。
- 增加 Web 或 MQTT 数据上报功能。
- 增加低功耗休眠和电池供电策略。
- 增加 SD 卡历史记录形成长期监测系统。
